# 08 - Evaluate SummaC and ROUGE-L

The greedy dev baseline and the three AlignScore-selected configurations are evaluated on the same ordered 619 abstracts. Raw candidate averages are not used.

## Fixed Configuration


In [1]:
from pathlib import Path
import json
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
shared_notebook = ROOT / 'notebooks/05_prepare_validation_ablation.ipynb'
shared_cells = json.loads(shared_notebook.read_text(encoding='utf-8'))['cells']
for shared_index, shared_cell in enumerate(shared_cells):
    if shared_cell['cell_type'] == 'code' and 'ablation-shared' in shared_cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(shared_cell['source']), str(shared_notebook) + ':cell' + str(shared_index), 'exec'), globals())
ablation = SummarizationAblation(CONFIG)
ablation.validate_run()


## Evaluation implementation
SummaC-ZS vitc compares summaries against original abstracts. ROUGE-L F1 uses stemming and the maximum over the corresponding human TLDR references. Metric snapshots are pinned and loaded offline; scores save per abstract.


In [2]:
def evaluate_summaries(self):
    self.validate_run()
    self.configure_metric_environment()
    import torch
    from summac.model_summac import SummaCZS
    from rouge_score.rouge_scorer import RougeScorer
    from tqdm.auto import tqdm
    scorer = SummaCZS(**self.c['summac'], device=self.c['device'], use_cache=False, imager_load_cache=False)
    rouge = RougeScorer(['rougeL'], use_stemmer=self.c['rouge']['use_stemmer'])
    for label in ['baseline', *[f'penalty_{p}' for p in self.c['penalties']]]:
        src = self.out / ('val_baseline.csv' if label == 'baseline' else f'reranked/{label}.csv')
        data = self.load_completed_output(src)
        get_stage_manifest = self.get_stage_manifest('evaluate_' + label, [src])
        dest = self.out / f'scores/{label}_scores.csv'
        rows = self.load_progress(dest, self.data, get_stage_manifest)
        for i in tqdm(range(len(rows), len(self.data)), desc=f'SummaC/ROUGE {label}'):
            text = data[i]['summary' if label == 'baseline' else 'selected_summary']
            require(bool(scorer.imager.split_sentences(text)), 'Empty summary sentences')
            with torch.inference_mode():
                score = float(scorer.score([self.data[i]['abstract']], [text], batch_size=8)['scores'][0])
            valid_scores([score], 1)
            r = max(rouge.score(ref, text)['rougeL'].fmeasure for ref in self.data[i]['human_tldrs'])
            rows.append(dict(id=self.ids[i], summary=text, summac=score, rougeL=r))
            self.save_progress(dest, rows, get_stage_manifest)
            scorer.imager.cache.clear()


SummarizationAblation.evaluate_summaries = evaluate_summaries


## Metric scoring
Outputs: one complete scores CSV for each of the four methods.


In [3]:
ablation.evaluate_summaries()


SummaC/ROUGE baseline:   0%|          | 0/619 [00:00<?, ?it/s]

SummaC/ROUGE penalty_0.5:   0%|          | 0/619 [00:00<?, ?it/s]

SummaC/ROUGE penalty_1.0:   0%|          | 0/619 [00:00<?, ?it/s]

SummaC/ROUGE penalty_2.0:   0%|          | 0/619 [00:00<?, ?it/s]

## Evaluation coverage
Every method requires 619 ordered scores before analysis.


In [4]:
for label in ['baseline', *[f'penalty_{p}' for p in CONFIG['penalties']]]:
    rows = ablation.load_completed_output(ablation.out / f'scores/{label}_scores.csv')
    print(f'{label}: {len(rows)} evaluated summaries')


baseline: 619 evaluated summaries
penalty_0.5: 619 evaluated summaries
penalty_1.0: 619 evaluated summaries
penalty_2.0: 619 evaluated summaries
